### Installing Libraries

In [7]:
pip install sentence-transformers scikit-learn numpy presidio-analyzer presidio-anonymizer

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install lingua-language-detector

  Obtaining dependency information for lingua-language-detector from https://files.pythonhosted.org/packages/42/15/8bf4abfe7147649c1f18521a65c790dcc646e2cc56e26bd02f5217d6b1d1/lingua_language_detector-2.0.2-cp312-cp312-macosx_11_0_arm64.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.2/349.2 kB 6.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.1/74.1 MB 6.8 MB/s eta 0:00:0000:0100:01m

[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from lingua import Language, LanguageDetectorBuilder

detector = LanguageDetectorBuilder.from_all_languages().build()

In [ ]:
# - NLP Mental Health Conversations (2022 samples).
# - Mental Health Conversational Data - Intents - Kaggel (200 samples).
# - Hope therapy conversations - Long multi turn conversations (8 samples but with on average 60 turns).
# - jerryjalapeno/nart-100k-synthetic (150 samples).
# - CalebE/new_mental_health_conversations_all1 (200 samples).
# - Gragroo/psyV0 (800 samples).

In [8]:
!pip3 install datasets

### Data Cleaning Utils Functions

In [3]:
import pandas as pd
from transformers import pipeline, AutoTokenizer

def filter_toxic_samples(csv_path, num_samples, threshold=0.1):
    """
    Extract samples from a train.csv file, removing duplicates based on row[0]
    and filtering out toxic content based on toxic-bert model.
    
    Args:
        csv_path (str): Path to the train.csv file
        num_samples (int): Number of samples to extract
        threshold (float): Toxicity threshold (default: 0.1)
    
    Returns:
        list: Filtered data without toxic elements
    """
    # Read the CSV file
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"Error reading CSV file: {e}")
        return []
    
    # Shuffle the dataframe
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    # Initialize the toxic-bert classifier
    classifier = pipeline("text-classification", model="unitary/toxic-bert")
    
    # Get the tokenizer and max length
    tokenizer = AutoTokenizer.from_pretrained("unitary/toxic-bert")
    max_length = tokenizer.model_max_length
    
    # Initialize collection variables
    result_data = []
    seen_row0 = set()
    toxic_elements = []
    
    # Process rows until we get the required number of samples or exhaust the dataframe
    row_index = 0
    remove_duplicate_user_questions = 0
    
    while len(result_data) < num_samples and row_index < len(df):
        
        try:
            row0 = str(df.iloc[row_index, 0])
            row1 = str(df.iloc[row_index, 1])
            
            # Skip if row0 already seen
            if row0 in seen_row0:
                row_index += 1
                remove_duplicate_user_questions += 1
                continue
            
            # Truncate texts to max length
            row0_tokens = tokenizer.encode(row0, add_special_tokens=True, max_length=max_length, truncation=True)
            row0_truncated = tokenizer.decode(row0_tokens, skip_special_tokens=True)
            
            row1_tokens = tokenizer.encode(row1, add_special_tokens=True, max_length=max_length, truncation=True)
            row1_truncated = tokenizer.decode(row1_tokens, skip_special_tokens=True)
            
            # Check toxicity
            result_row0 = classifier(row0_truncated)[0]
            toxic_row0 = result_row0['label'] == 'toxic' and result_row0['score'] > threshold
            
            result_row1 = classifier(row1_truncated)[0]
            toxic_row1 = result_row1['label'] == 'toxic' and result_row1['score'] > threshold
            
            # If any part is toxic, add to toxic elements and skip
            if toxic_row0:
                toxic_text = row0
                toxic_elements.append((toxic_text, result_row0['score'], "row0"))
                print(f"Toxic row0 (score: {result_row0['score']:.4f}): {toxic_text}")
            
            if toxic_row1:
                toxic_text = row1
                toxic_elements.append((toxic_text, result_row1['score'], "row1"))
                print(f"Toxic row1 (score: {result_row1['score']:.4f}): {toxic_text}")
            
            # Add to result if neither is toxic
            if not (toxic_row0 or toxic_row1):
                result_data.append([row0, row1])
                seen_row0.add(row0)
            
            row_index += 1
            
        except Exception as e:
            print(f"Error processing row {row_index}: {e}")
            row_index += 1
    
    print(f"\nTotal samples collected: {len(result_data)}")
    print(f"Total toxic elements removed: {len(toxic_elements)}")
    print(f"Removed duplicate user questions: {remove_duplicate_user_questions}")
    
    return result_data

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Language Detection Init

In [4]:
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine

# Initialize the analyzer and anonymizer
analyzer = AnalyzerEngine()
anonymizer = AnonymizerEngine()

def detect_language(text):
    if not text or len(text.strip()) == 0:
        return "UNKNOWN"
    
    language = detector.detect_language_of(text)
    if language is None:
        return "UNKNOWN"
    
    return language.name

In [5]:
def anonymize_text(text):
    results = analyzer.analyze(
        text=text,
        entities=["PERSON", "DATE_TIME", "LOCATION", "PHONE_NUMBER", "EMAIL_ADDRESS"],
        language="en"
    )
    anonymized_text = anonymizer.anonymize(text=text, analyzer_results=results)

    return results, anonymized_text.text

In [6]:
SYSTEM_PROMPT = """Provide support and understanding to individuals experiencing emotional and psychological challenges. Focus on conversation style, insights and therapy."""

In [32]:
import csv
import re
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

def format_text(text):
    # Add space after punctuation if missing
    text = re.sub(r'([.!?])([A-Za-z])', r'\1 \2', text)
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)
    
    # Trim spaces at the start/end of sentences
    text = re.sub(r'\s([.!?])', r'\1', text)
    
    # Final trim
    return text.strip()

def clean_data(text):
    # Regex pattern matches 'http://' or 'https://' followed by non-whitespace characters.
    pattern = r'https?://\S+'
    cleaned_text = re.sub(pattern, '', text.replace('\xa0', ' '))
    
    return format_text(cleaned_text)

def deduplicate_data(data, similarity_threshold=0.95):
    """
    Deduplicate rows in data based on separate similarity checks for user and assistant texts.
    A pair is considered duplicate if both the user and assistant message similarities exceed the threshold.
    
    Args:
        data: List of dicts, each containing a 'messages' key with user and assistant messages.
        similarity_threshold: Float threshold for cosine similarity to consider rows as duplicates.
    
    Returns:
        A deduplicated list of data rows.
    """
    # Extract user and assistant texts from the data
    texts_user = [row['messages'][1]['content'] for row in data]
    texts_assistant = [row['messages'][2]['content'] for row in data]
    
    # Load the SentenceTransformer model (you may choose another model if desired)
    model = SentenceTransformer('all-MiniLM-L6-v2')
    
    # Compute embeddings for user and assistant messages separately
    embeddings_user = model.encode(texts_user)
    embeddings_assistant = model.encode(texts_assistant)

    # Compute cosine similarity matrices for both sets of embeddings
    sim_matrix_user = cosine_similarity(embeddings_user)
    sim_matrix_assistant = cosine_similarity(embeddings_assistant)
    
    duplicates = set()
    
    print("Checking for duplicate rows based on similarity in both user and assistant messages:")
    for i in range(len(data)):
        for j in range(i + 1, len(data)):
            if (sim_matrix_user[i][j] >= similarity_threshold and 
                sim_matrix_assistant[i][j] >= similarity_threshold):
                print(f"\nPotential duplicate pair (Row {i} and Row {j}):")
                print(f"User message Row {i}: {texts_user[i]}")
                print(f"User message Row {j}: {texts_user[j]}")
                print(f"Assistant message Row {i}: {texts_assistant[i]}")
                print(f"Assistant message Row {j}: {texts_assistant[j]}")
                # print(f"User similarity: {sim_matrix_user[i][j]:.4f}")
                # print(f"Assistant similarity: {sim_matrix_assistant[i][j]:.4f}")
                # print(f"User embedding Row {i}: {embeddings_user[i]}")
                # print(f"User embedding Row {j}: {embeddings_user[j]}")
                # print(f"Assistant embedding Row {i}: {embeddings_assistant[i]}")
                # print(f"Assistant embedding Row {j}: {embeddings_assistant[j]}")
                duplicates.add(j)
    
    deduped_data = [row for idx, row in enumerate(data) if idx not in duplicates]
    print(f"\nRemoved {len(duplicates)} duplicate rows out of {len(data)} total rows.")
    return deduped_data

def clean_anonymize_and_create_messages(userText, assistantText, skip_lang=False, skip_anon=False):
    messages = []
    user_data = clean_data(userText)
    assistant_data = clean_data(assistantText)

    if not skip_anon:
        user_results, user_data_anonymized = anonymize_text(user_data)
        asisstant_results, assistant_data_anonymized = anonymize_text(assistant_data)
        user_data = user_data_anonymized
        assistant_data = assistant_data_anonymized

    lang_user = detect_language(user_data)
    lang_assistant = detect_language(assistant_data)

    if (lang_user != "ENGLISH" or lang_assistant != "ENGLISH") and not skip_lang:
        print(user_data)
        print(assistant_data)
        print('Skipping because it is not english text')
        return None

    CURRENT_SYSTEM_PROMPT = SYSTEM_PROMPT
    # if len(user_results) != 0 or len(asisstant_results) != 0:
    #     CURRENT_SYSTEM_PROMPT += SYSTEM_PROMPT_ADDITION_FOR_ANONYMIZATION

    messages.append({
        'role': 'system',
        'content': CURRENT_SYSTEM_PROMPT
    })

    messages.append({
        'role': 'user',
        'content': user_data
    })
    messages.append({
        'role': 'assistant',
        'content': assistant_data
    })

    return messages

### Load NLP Mental Health Conversations and clean it - format it for fine tuning

#### Filter toxic examples

In [18]:
nlp_mental_health_convs = filter_toxic_samples("data/train.csv", 6300, 0.2)

Device set to use mps:0


Toxic row0 (score: 0.2261): Maybe this is a stupid question, but I sometimes don't know what's real or not. If feel at times like everyone's lying. How do I know if God is one of those lies?
Toxic row0 (score: 0.6930): I need to speak to someone about sexual addiction and binge eating immediately.
Toxic row0 (score: 0.3906): We went out had great sex and I was really liking her. Then one day she says we should just be friends, but I can't stop thinking about her. She's always on mind and I want her back.
Toxic row0 (score: 0.2301): I feel like I have to be promiscuous in order to keep people around? It started after I got raped by my ex-boyfriend.
Toxic row0 (score: 0.6217): My mom is always bossy and treats me like a child even though I'm in my 20s. She argues with me over stupid stuff.
Toxic row0 (score: 0.4816): I love my girlfriend so much. I get an erection even just thinking about her or seeing her.  But the two times we tried to have sex I couldn't get an erection.  We've only h

In [42]:
parsed_nlp_mental_health_convs = []

for idx, row in enumerate(nlp_mental_health_convs):
    if not row[0] or not row[1]:
        continue
    # Process the row normally if not skipped
    messages = clean_anonymize_and_create_messages(row[0], row[1])

    if messages == None:
        continue

    parsed_nlp_mental_health_convs.append({
        "messages": messages
    })

nlp_mental_health_convs_deduped = deduplicate_data(parsed_nlp_mental_health_convs)

I've had posttraumatic stress disorder for years without my parents ever finding out. I want to overcome it, but it’s so vivid, it’s like it’s happening again. I'm scared and paranoid. I have depression, which I have been struggling with since a young age.
I understand that at times it’s difficult to share with our parents what we have been trough, due to fear of judgment or punishment; but I have noticed that keeping our experiences secret, intensifies them. And Post Traumatic Stress Disorder can haunt us for a long time if we do not learn to process the memories that cause those symptoms. If you have been victim of a traumatic event that you are not prepared to share with your parents, it’s important that you seek help with a counselor, therapist or psychologist. Trauma Focused Cognitive Behavioral Therapy is a great technique to cope with physical symptoms, you could also use Narrative therapy, Creative Therapy or Journaling as a way to express memories and process them with your th

In [43]:
print(nlp_mental_health_convs_deduped[0])

{'messages': [{'role': 'system', 'content': 'Provide support and understanding to individuals experiencing emotional challenges. Each user message is an example of existing conversational data and the assistant response is to teach you how to be more human.'}, {'role': 'user', 'content': "I've hit my head on walls and floors ever since I was young. I sometimes still do it but I don't exactly know why, I have anxiety and I had a rough childhood but now I'll start to hit my head and sometimes not realize it but I don't know how to stop or even why I'm doing it. How can I help myself to change my behavior?"}, {'role': 'assistant', 'content': 'The best way to handle anxiety of this level is with a combination of appropriate medication given to you by a medical doctor, and therapy to help you understand the thoughts, feelings, and behaviors that are causing the anxiety. This is not something that anyone should just “white knuckle” and try to get through on their own with no help. Cognitive 

#### Format data for LLM filtering and rephrasing, this would replace tags and eliminate confusion for the LLM

In [42]:
import uuid

def create_data_for_gpt_batch(dataset, additional_rules=None, should_remove_names=True):
    data_to_filter_request = []

    for idx, row in enumerate(dataset):
        messages = row['messages']
        # Augment the user message to do the filtering
        user_message = messages[1]["content"]
        assistant_message = messages[2]["content"]
        actual_conversation = f"User: {user_message}\nAssistant: {assistant_message}"

        # Remove the assistant message
        data_to_filter_request.append({
            "custom_id": f"filter-{idx}-{uuid.uuid4()}",
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": "gpt-4o-mini",
                "messages": [
                    {
                        "role": "system",
                        "content": "Your role is to categorize and reformulate user assistant conversations."
                    },
                    {
                        "role": "user",
                        "content":  f"""You have user-assistant psychological conversation data. Categorize and reformulate each pair clearly as follows:
    REMOVE for conversations containing:
    1. Format issues: Nonsense text, weird characters, repeating words, sentences out of place, incomplete conversations, unusual punctuation
    2. Non-evidence-based approaches: Psychological advice without scientific support
    3. Harmful guidance: Content dismissing serious concerns or potentially endangering clients
    4. Inappropriate framing: Using primarily non-psychological frameworks for mental health issues
    5. Unprofessional communication: Judgmental language, rude commentary, excessive personal disclosures
    6. Inadequate responses: Failing to address client concerns, vague advice, irrelevant replies
    7. Unethical content: Violating professional ethics or promoting harmful practices
    8. Recommendations for specific books/websites.
    9. References to specific medications without medical expertise
    10. Promoting or advertising: "you can find more information on my website at..." And so on.
    {additional_rules if additional_rules != None else ""}

    KEEP for conversations demonstrating:
    1. Evidence-based approaches with scientific support
    2. Professional boundaries and appropriate clinical judgment
    3. Client empowerment and respect for autonomy
    4. Ethical therapeutic practice and clear, focused guidance

    {"""
    Reformulation Rules (if needed):
    * Replace <PERSON> in assistant messages referring to the speaker with "Atlas".
    * Remove <PERSON> tags referring to the user or others. For others you could generalise depending on context (your girlfriend, your family, schoolmates) if you know, if not, remove.
    * Replace <DATE_TIME> with general terms ("today", "soon", "next week", randomly).
    * Replace phone numbers with neutral phrases like "call someone" or "call for help".
    * If there are any other names in the conversation that are not replaced with <PERSON> please correct and remove those names.
    * Important! Try to maintain the tone of the conversation.""" if should_remove_names else ""}

    Respond with a JSON Output:{{\“category\”: “REMOVE” or “KEEP”, “reformulation”: {{"user": reformulated_user_message, "assistant": reformulated_assistant_message}}}}

    Do not provide reformulation if category is REMOVE, only for KEEP.
    Do not add anything else but the JSON output.

    Conversation:
    {actual_conversation}"""
                    }
                ]
            }
        })

    return data_to_filter_request


#### Call OpenAI api to prepare batches for processing

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_KEY"

In [47]:
import json

data_to_filter_request = create_data_for_gpt_batch(nlp_mental_health_convs_deduped)

with open("to_filter_dataset.jsonl", 'w') as f:
    for request in data_to_filter_request:
        f.write(json.dumps(request) + '\n')

In [48]:
from openai import OpenAI
client = OpenAI()

batch_input_file = client.files.create(
    file=open("to_filter_dataset.jsonl", "rb"),
    purpose="batch"
)

print(batch_input_file)

FileObject(id='file-XSv4sbLyqGVgipnxWS2ofj', bytes=3299983, created_at=1741651459, filename='to_filter_dataset.jsonl', object='file', purpose='batch', status='processed', status_details=None, expires_at=None)


In [49]:
from openai import OpenAI
client = OpenAI()

batch_input_file_id = batch_input_file.id
client.batches.create(
    input_file_id=batch_input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h"
)

Batch(id='batch_67cf7e0c68c48190a6e694712efb5811', completion_window='24h', created_at=1741651468, endpoint='/v1/chat/completions', input_file_id='file-XSv4sbLyqGVgipnxWS2ofj', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1741737868, failed_at=None, finalizing_at=None, in_progress_at=None, metadata=None, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0))

In [55]:
from openai import OpenAI
client = OpenAI()

batch = client.batches.retrieve("batch_67cf7e0c68c48190a6e694712efb5811")
print(batch)

Batch(id='batch_67cf7e0c68c48190a6e694712efb5811', completion_window='24h', created_at=1741651468, endpoint='/v1/chat/completions', input_file_id='file-XSv4sbLyqGVgipnxWS2ofj', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1741655051, error_file_id=None, errors=None, expired_at=None, expires_at=1741737868, failed_at=None, finalizing_at=1741654974, in_progress_at=1741651469, metadata=None, output_file_id='file-QmHe9kxyHyhtEeSPA14vWk', request_counts=BatchRequestCounts(completed=819, failed=0, total=819))


#### Parse the data and filter it for including in the fine tuning dataset

In [53]:
import json
import re

def parse_batch_responses(response_text):
    """
    Parse the batch API responses and extract the reformulated conversations,
    preserving the assistant responses as JSON objects.
    
    Args:
        response_text (str): The raw text of the batch API responses
        
    Returns:
        list: A list of dictionaries containing the parsed conversations
    """
    # Split the text into individual response objects
    # Each response object should be a complete JSON object
    response_objects = []
    
    # Try to parse the entire response as a JSON array first
    try:
        response_objects = json.loads(f"[{response_text.replace('}{', '},{')}]")
    except json.JSONDecodeError:
        # If that fails, try to extract individual JSON objects
        json_pattern = r'(\{.*?\}\n)'
        matches = re.findall(json_pattern, response_text, re.DOTALL)
        
        if not matches:
            # Try alternative pattern if the first one doesn't work
            json_pattern = r'(\{.*?\})'
            matches = re.findall(json_pattern, response_text, re.DOTALL)
            
        for json_str in matches:
            try:
                response_obj = json.loads(json_str.strip())
                response_objects.append(response_obj)
            except json.JSONDecodeError:
                continue
    
    parsed_data = []
    
    for response_obj in response_objects:
        try:
            # Extract the custom_id which contains our filter ID
            custom_id = response_obj.get('custom_id', '')
            
            # Get the response content
            completion_content = None
            try:
                completion_content = response_obj.get('response', {}).get('body', {}).get('choices', [{}])[0].get('message', {}).get('content', '')
            except (KeyError, IndexError):
                pass
                
            # If that didn't work, try the new structure with 'output' field
            if not completion_content:
                completion_content = response_obj.get('output', '')
                
            if not completion_content:
                continue
                
            # Clean up any markdown formatting
            if completion_content.startswith('```json'):
                completion_content = completion_content.replace('```json', '').replace('```', '').strip()
            
            # Try to parse the completion content as JSON
            try:
                result = json.loads(completion_content)
                
                # Only include conversations marked as KEEP
                if result.get('category') == 'KEEP':
                    user_content = result.get('reformulation', {}).get('user', '')
                    assistant_content = result.get('reformulation', {}).get('assistant', '')
                    
                    parsed_data.append({
                        'id': custom_id,
                        'category': 'KEEP',
                        'user': user_content,
                        'assistant': assistant_content,
                        'raw_json': result  # Include the entire raw JSON for reference
                    })
            except json.JSONDecodeError:
                # If we can't parse the JSON, try to extract using regex
                category_match = re.search(r'"category":\s*"(KEEP|REMOVE)"', completion_content)
                if category_match and category_match.group(1) == 'KEEP':
                    user_match = re.search(r'"user":\s*"(.*?)"', completion_content, re.DOTALL)
                    assistant_match = re.search(r'"assistant":\s*"(.*?)"', completion_content, re.DOTALL)
                    
                    user_content = user_match.group(1) if user_match else ''
                    assistant_content = assistant_match.group(1) if assistant_match else ''
                    
                    parsed_data.append({
                        'id': custom_id,
                        'category': 'KEEP',
                        'user': user_content,
                        'assistant': assistant_content,
                        'original_content': completion_content  # Store original content for debugging
                    })
                    
        except Exception as e:
            print(f"Error processing response: {e}")
            continue
    
    return parsed_data

In [57]:
from openai import OpenAI
client = OpenAI()

file_response = client.files.content("file-QmHe9kxyHyhtEeSPA14vWk")
print(file_response.text)

{"id": "batch_req_67cf8bbf01bc8190a002a016517c60bd", "custom_id": "filter-0-a8c0d76a-2124-49cc-9a4c-ae5ba8029742", "response": {"status_code": 200, "request_id": "92a6ff9fe8b7c8f793f3393104df837a", "body": {"id": "chatcmpl-B9hOjTAhuXloFf5Y63zNj6DbZdUoe", "object": "chat.completion", "created": 1741651473, "model": "gpt-4o-mini-2024-07-18", "choices": [{"index": 0, "message": {"role": "assistant", "content": "{\n  \"category\": \"KEEP\",\n  \"reformulation\": {\n    \"user\": \"I've hit my head on walls and floors ever since I was young. I sometimes still do it but I don't exactly know why. I have anxiety and a rough childhood. I want to change my behavior, but I don't know how to stop or even why I'm doing it.\",\n    \"assistant\": \"The best way to handle anxiety of this level is through a combination of appropriate medical guidance and therapy. Engaging in Cognitive Behavioral Therapy (CBT) can help you understand the thoughts, feelings, and behaviors that contribute to your anxiety

In [58]:
parsed_data = parse_batch_responses(file_response.text)

In [60]:
keep_data = []

for data in parsed_data:
    if data['category'] == 'KEEP':
        keep_data.append(data)

In [64]:
keep_data_formatted = []

for keep_d in keep_data:
    keep_data_formatted.append({
        "messages": [{
            "role": "system",
            "content": SYSTEM_PROMPT
        }, {
            "role": "user",
            "content": keep_d['user']
        }, {
            "role": "assistant",
            "content": keep_d['assistant']
        }]
    })

In [66]:
with open("nlp_mental_health_filtered_cleaned_rephreased", 'w') as file:
    for item in keep_data_formatted:
        json_line = json.dumps(item)
        file.write(json_line + '\n')

In [27]:
# Everything above goes into one function so we can use it later

def parse_response_and_write_data(file_response, output_file, CUSTOM_SYSTEM_PROMPT=None):
    parsed_data = parse_batch_responses(file_response.text)

    keep_data = []

    for data in parsed_data:
        if data['category'] == 'KEEP':
            keep_data.append(data)

    keep_data_formatted = []

    for keep_d in keep_data:
        keep_data_formatted.append({
            "messages": [{
                "role": "system",
                "content": CUSTOM_SYSTEM_PROMPT if CUSTOM_SYSTEM_PROMPT != None else SYSTEM_PROMPT
            }, {
                "role": "user",
                "content": keep_d['user']
            }, {
                "role": "assistant",
                "content": keep_d['assistant']
            }]
        })

    with open(output_file, 'w') as file:
        for item in keep_data_formatted:
            json_line = json.dumps(item)
            file.write(json_line + '\n')

### Prepare the Health Conversation Data - Intents

#### Some utils for dealing with filtering conversations

In [39]:
import random

def filter_toxic_conversations(conversations, num_samples, threshold=0.1, random_seed=42):
    """
    Filter a list of conversation objects, removing toxic content based on toxic-bert model.
    
    Args:
        conversations (list): List of conversation objects with 'messages' field containing exchanges
        num_samples (int): Number of conversations to extract
        threshold (float): Toxicity threshold (default: 0.1)
        random_seed (int): Random seed for shuffling (default: 42)
    
    Returns:
        list: Filtered conversations without toxic elements
    """
    # Set random seed for reproducibility
    random.seed(random_seed)
    
    # Shuffle the conversations
    shuffled_conversations = random.sample(conversations, len(conversations))
    
    # Initialize the toxic-bert classifier
    classifier = pipeline("text-classification", model="unitary/toxic-bert")
    
    # Get the tokenizer and max length
    tokenizer = AutoTokenizer.from_pretrained("unitary/toxic-bert")
    max_length = tokenizer.model_max_length
    
    # Initialize collection variables
    result_conversations = []
    seen_conversations = set()
    toxic_elements = []
    
    # Process conversations until we get the required number of samples or exhaust the list
    conv_index = 0
    removed_duplicate_conversations = 0
    
    while len(result_conversations) < num_samples and conv_index < len(shuffled_conversations):
        try:
            conversation = shuffled_conversations[conv_index]
            messages = conversation.get('messages', [])
            
            # Create a unique identifier for this conversation
            # Using the first user message as the identifier
            user_messages = [msg['content'] for msg in messages if msg['role'] == 'user']
            if not user_messages:
                conv_index += 1
                continue
                
            conv_id = user_messages[0]
            
            # Skip if conversation already seen
            if conv_id in seen_conversations:
                removed_duplicate_conversations += 1
                conv_index += 1
                continue
            
            is_toxic = False
            
            # Check each message for toxicity
            for msg in messages:
                if msg['role'] not in ['user', 'assistant']:
                    continue
                
                content = str(msg['content'])
                
                # Truncate text to max length
                content_tokens = tokenizer.encode(content, add_special_tokens=True, max_length=max_length, truncation=True)
                content_truncated = tokenizer.decode(content_tokens, skip_special_tokens=True)
                
                # Check toxicity
                result = classifier(content_truncated)[0]
                toxic = result['label'] == 'toxic' and result['score'] > threshold
                
                if toxic:
                    toxic_text = content
                    toxic_elements.append((toxic_text, result['score'], msg['role']))
                    print(f"Toxic {msg['role']} (score: {result['score']:.4f}): {toxic_text[:100]}..." if len(toxic_text) > 100 else toxic_text)
                    is_toxic = True
                    break
            
            # Add to result if not toxic
            if not is_toxic:
                result_conversations.append(conversation)
                seen_conversations.add(conv_id)
            
            conv_index += 1
            
        except Exception as e:
            print(f"Error processing conversation {conv_index}: {e}")
            conv_index += 1
    
    print(f"\nTotal conversations collected: {len(result_conversations)}")
    print(f"Total toxic elements removed: {len(toxic_elements)}")
    print(f"Removed duplicate conversations: {removed_duplicate_conversations}")
    
    return result_conversations

#### Create convs and filter out toxic data

In [70]:
import json
import random

# Mental Health Conversational Data - Intents - Kaggel
# Intents dataset
def read_intents_dataset(filePath):
    with open(filePath, 'r', encoding='utf-8') as f:
        intents_data = json.load(f)

        intentsData = []
        intents = intents_data.get("intents", [])
        
        for intent in intents:
            patterns = intent.get("patterns", [])
            responses = intent.get("responses", [])
            if not patterns:
                continue  # Skip if no patterns provided
            
            pairs = []
            n_patterns = len(patterns)
            n_responses = len(responses)
            
            if n_responses <= n_patterns:
                # One-to-one pairing; discard extra patterns if any
                pairs = list(zip(patterns, responses))
            else:
                # Pair first n_patterns responses one-to-one and pair extra responses with a random pattern
                pairs = list(zip(patterns, responses[:n_patterns]))
                extra_responses = responses[n_patterns:]
                for resp in extra_responses:
                    random_pattern = random.choice(patterns)
                    pairs.append((random_pattern, resp))

            # Build the messages structure for each (pattern, response) pair
            for user_text, assistant_text in pairs:
                if not user_text or not assistant_text:
                    continue

                messages = clean_anonymize_and_create_messages(user_text, assistant_text, skip_lang=True)
                if messages != None:
                    intentsData.append({
                        'messages': messages
                    })
            
        # Deduplicate the generated training examples
        return deduplicate_data(intentsData)

In [71]:
mental_health_convs_intents = read_intents_dataset('data/intents.json')

Checking for duplicate rows based on similarity in both user and assistant messages:

Removed 0 duplicate rows out of 152 total rows.


In [73]:
mental_health_convs_intents_non_toxic = filter_toxic_conversations(mental_health_convs_intents, 3000, 0.2)

Device set to use mps:0


I hate you
I want to kill myself
You hate me
My mom died
You're just some robot. How would you know?
I don't like you
Are you stupid?
I know you hate me
Just shut up

Total conversations collected: 135
Total toxic elements removed: 9
Removed duplicate conversations: 8


In [76]:
mental_health_convs_intents_therapy = read_intents_dataset('data/intents-therapy.json')

Checking for duplicate rows based on similarity in both user and assistant messages:

Removed 0 duplicate rows out of 39 total rows.


In [77]:
mental_health_convs_intents_therapy_non_toxic = filter_toxic_conversations(mental_health_convs_intents_therapy, 3000, 0.2)

Device set to use mps:0



Total conversations collected: 39
Total toxic elements removed: 0
Removed duplicate conversations: 0


In [97]:
intents_ds = mental_health_convs_intents_non_toxic + mental_health_convs_intents_therapy_non_toxic
print(len(intents_ds))

174


In [100]:
import json

data_to_filter_request = create_data_for_gpt_batch(intents_ds)

with open("to_filter_dataset_intents.jsonl", 'w') as f:
    for request in data_to_filter_request:
        f.write(json.dumps(request) + '\n')

In [102]:
from openai import OpenAI
client = OpenAI()

batch_input_file = client.files.create(
    file=open("to_filter_dataset_intents.jsonl", "rb"),
    purpose="batch"
)

print(batch_input_file)

FileObject(id='file-HPysQzdi2Yo2X4r9Teshg7', bytes=535021, created_at=1741718736, filename='to_filter_dataset_intents.jsonl', object='file', purpose='batch', status='processed', status_details=None, expires_at=None)


In [103]:
from openai import OpenAI
client = OpenAI()

batch_input_file_id = batch_input_file.id
client.batches.create(
    input_file_id=batch_input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h"
)

Batch(id='batch_67d084e53dfc8190b49b1f057c74c6b6', completion_window='24h', created_at=1741718757, endpoint='/v1/chat/completions', input_file_id='file-HPysQzdi2Yo2X4r9Teshg7', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1741805157, failed_at=None, finalizing_at=None, in_progress_at=None, metadata=None, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0))

In [193]:
### Rerun this to get the progress

from openai import OpenAI
client = OpenAI()

batch = client.batches.retrieve("batch_67d084e53dfc8190b49b1f057c74c6b6")
print(batch)

Batch(id='batch_67d084e53dfc8190b49b1f057c74c6b6', completion_window='24h', created_at=1741718757, endpoint='/v1/chat/completions', input_file_id='file-HPysQzdi2Yo2X4r9Teshg7', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1741727475, error_file_id=None, errors=None, expired_at=None, expires_at=1741805157, failed_at=None, finalizing_at=1741727460, in_progress_at=1741718757, metadata=None, output_file_id='file-YQHUdPCR7MPKKBGUyq9G8k', request_counts=BatchRequestCounts(completed=174, failed=0, total=174))


##### Process LLM data (filter based on category, and update the messages with rephrased ones)

In [194]:
from openai import OpenAI
client = OpenAI()

file_response = client.files.content("file-YQHUdPCR7MPKKBGUyq9G8k")
print(file_response.text)

{"id": "batch_req_67d0a6e44e548190a84134c184b59da0", "custom_id": "filter-0-09688f01-a987-4f41-9206-e78a983cbe33", "response": {"status_code": 200, "request_id": "7c5ccc7425bcfc9c9c13ba44abc30ca6", "body": {"id": "chatcmpl-BA0VysU3UkeHTnpoprkYiiQMjnLON", "object": "chat.completion", "created": 1741724958, "model": "gpt-4o-mini-2024-07-18", "choices": [{"index": 0, "message": {"role": "assistant", "content": "{\n  \"category\": \"REMOVE\",\n  \"reformulation\": {}\n}", "refusal": null, "annotations": []}, "logprobs": null, "finish_reason": "stop"}], "usage": {"prompt_tokens": 537, "completion_tokens": 17, "total_tokens": 554, "prompt_tokens_details": {"cached_tokens": 0, "audio_tokens": 0}, "completion_tokens_details": {"reasoning_tokens": 0, "audio_tokens": 0, "accepted_prediction_tokens": 0, "rejected_prediction_tokens": 0}}, "service_tier": "default", "system_fingerprint": "fp_06737a9306"}}, "error": null}
{"id": "batch_req_67d0a6e463d4819099e8c9325607b112", "custom_id": "filter-1-16

In [196]:
parse_response_and_write_data(file_response, "rephrased_cleaned_parsed/mental_health_conversational_intents.jsonl")

### Process Hope Conversations Dataset

In [13]:
import os
import csv

def process_csv_directory(directory_path):
    """
    Reads every CSV file in the given directory and creates a multi-turn conversation
    for each file by taking the third entry (index 2) from each row.
    The conversation alternates roles: the first message is from the user, then assistant,
    then user, then assistant, and so on.
    
    If the last message is from the user, it is removed so that the conversation ends with an assistant message.
    
    Args:
        directory_path (str): The path to the directory containing CSV files.
    
    Returns:
        List[dict]: A list of conversation dictionaries, each with a "messages" key.
    """
    conversations = []
    # Loop over all files in the directory
    for filename in os.listdir(directory_path):
        if filename.lower().endswith('.csv'):
            filepath = os.path.join(directory_path, filename)
            messages = [{
                "role": "system",
                "content": SYSTEM_PROMPT
            }]
            # Open and read the CSV file
            with open(filepath, 'r', encoding='utf-8') as csvfile:
                reader = csv.reader(csvfile)
                # Define role order: first user, then assistant, then alternating.
                roles = ['assistant', 'user']
                role_index = 0
                for index, row in enumerate(reader):
                    if index == 0:
                        continue  # Skip header row if exists.
                    # Ensure the row has at least three entries.
                    if len(row) < 3:
                        continue
                    # Get the third entry from the row.
                    content = row[2].strip()

                    content_cleaned = clean_data(content)
                    _, anonmized_text = anonymize_text(content_cleaned)

                    messages.append({
                        'role': roles[role_index],
                        'content': anonmized_text
                    })
                    # Alternate the role for the next message.
                    role_index = 1 - role_index
            # Remove the last message if it's from the user.
            if len(messages) > 1 and messages[-1]['role'] == 'user':
                messages.pop()
            conversations.append({"messages": messages})
    return conversations

In [14]:
hope_convs = process_csv_directory('data/hope_therapy_conversations')

#### Define an util for multi turn conv filtering of toxicity

In [37]:
import random
from transformers import pipeline, AutoTokenizer

def filter_toxic_conversations_multi(conversations, num_samples, threshold=0.1, random_seed=42):
    """
    Filter a list of conversation objects, removing toxic content based on toxic-bert model.
    For multi-turn conversations, it removes the user-assistant pair instead of the whole conversation.
    
    Args:
        conversations (list): List of conversation objects with 'messages' field containing exchanges
        num_samples (int): Number of conversations to extract
        threshold (float): Toxicity threshold (default: 0.1)
        random_seed (int): Random seed for shuffling (default: 42)
    
    Returns:
        list: Filtered conversations with toxic pairs removed
    """
    # Set random seed for reproducibility
    random.seed(random_seed)
    
    # Shuffle the conversations
    shuffled_conversations = random.sample(conversations, len(conversations))
    
    # Initialize the toxic-bert classifier
    classifier = pipeline("text-classification", model="unitary/toxic-bert")
    
    # Get the tokenizer and max length
    tokenizer = AutoTokenizer.from_pretrained("unitary/toxic-bert")
    max_length = tokenizer.model_max_length
    
    # Initialize collection variables
    result_conversations = []
    seen_conversations = set()
    toxic_elements = []
    removed_message_pairs = 0
    
    # Process conversations until we get the required number of samples or exhaust the list
    conv_index = 0
    removed_duplicate_conversations = 0
    
    while len(result_conversations) < num_samples and conv_index < len(shuffled_conversations):
        try:
            conversation = shuffled_conversations[conv_index]
            messages = conversation.get('messages', [])
            
            # Create a unique identifier for this conversation
            # Using the first user message as the identifier
            user_messages = [msg['content'] for msg in messages if msg['role'] == 'user']
            if not user_messages:
                conv_index += 1
                continue
                
            conv_id = user_messages[0]
            
            # Skip if conversation already seen
            if conv_id in seen_conversations:
                removed_duplicate_conversations += 1
                conv_index += 1
                continue
            
            # Create a clean copy of messages for processing
            clean_messages = []
            i = 0
            
            while i < len(messages):
                # Skip non-user and non-assistant messages
                if messages[i]['role'] not in ['user', 'assistant']:
                    clean_messages.append(messages[i])
                    i += 1
                    continue
                
                # Process potential user-assistant pairs
                current_msg = messages[i]
                next_msg = messages[i+1] if i+1 < len(messages) else None
                
                # Check if current message is toxic
                content = str(current_msg['content'])
                content_tokens = tokenizer.encode(content, add_special_tokens=True, max_length=max_length, truncation=True)
                content_truncated = tokenizer.decode(content_tokens, skip_special_tokens=True)
                
                current_result = classifier(content_truncated)[0]
                current_toxic = current_result['label'] == 'toxic' and current_result['score'] > threshold
                
                # Check if next message exists and is toxic
                next_toxic = False
                if next_msg and next_msg['role'] in ['user', 'assistant']:
                    next_content = str(next_msg['content'])
                    next_content_tokens = tokenizer.encode(next_content, add_special_tokens=True, max_length=max_length, truncation=True)
                    next_content_truncated = tokenizer.decode(next_content_tokens, skip_special_tokens=True)
                    
                    next_result = classifier(next_content_truncated)[0]
                    next_toxic = next_result['label'] == 'toxic' and next_result['score'] > threshold
                
                # Handle toxic cases based on role
                if current_toxic or next_toxic:
                    # Log the toxic element
                    if current_toxic:
                        toxic_elements.append((content, current_result['score'], current_msg['role']))
                        print(f"Toxic {current_msg['role']} (score: {current_result['score']:.4f}): {content[:100]}..." if len(content) > 100 else content)
                    
                    if next_toxic:
                        next_content = str(next_msg['content'])
                        toxic_elements.append((next_content, next_result['score'], next_msg['role']))
                        print(f"Toxic {next_msg['role']} (score: {next_result['score']:.4f}): {next_content[:100]}..." if len(next_content) > 100 else next_content)
                    
                    # User is toxic: delete user message and the following assistant message
                    if current_msg['role'] == 'user' and current_toxic:
                        i += 2 if next_msg and next_msg['role'] == 'assistant' else 1
                        removed_message_pairs += 1
                    # Assistant is toxic: delete assistant message and the previous user message
                    elif current_msg['role'] == 'assistant' and current_toxic:
                        # Remove the previous user message if it was added
                        if clean_messages and clean_messages[-1]['role'] == 'user':
                            clean_messages.pop()
                        i += 1
                        removed_message_pairs += 1
                    # Next message is toxic: handle based on its role
                    elif next_toxic:
                        if next_msg['role'] == 'user':
                            # Keep current, skip next and its following assistant if it exists
                            clean_messages.append(current_msg)
                            i += 3 if i+2 < len(messages) and messages[i+2]['role'] == 'assistant' else 2
                            removed_message_pairs += 1
                        else:  # next_msg['role'] == 'assistant'
                            # Skip both current and next
                            i += 2
                            removed_message_pairs += 1
                    
                else:
                    # No toxicity, add current message to clean messages
                    clean_messages.append(current_msg)
                    i += 1
            
            # Add the cleaned conversation to results if it has meaningful exchanges left
            if clean_messages:
                # Count user-assistant pairs in the cleaned conversation
                user_msgs = [msg for msg in clean_messages if msg['role'] == 'user']
                assistant_msgs = [msg for msg in clean_messages if msg['role'] == 'assistant']
                
                # Only add if there's at least one complete exchange
                if user_msgs and assistant_msgs:
                    cleaned_conversation = conversation.copy()
                    cleaned_conversation['messages'] = clean_messages
                    result_conversations.append(cleaned_conversation)
                    seen_conversations.add(conv_id)
            
            conv_index += 1
            
        except Exception as e:
            print(f"Error processing conversation {conv_index}: {e}")
            conv_index += 1
    
    print(f"\nTotal conversations collected: {len(result_conversations)}")
    print(f"Total toxic elements removed: {len(toxic_elements)}")
    print(f"Removed message pairs: {removed_message_pairs}")
    print(f"Removed duplicate conversations: {removed_duplicate_conversations}")
    
    return result_conversations

In [16]:
hope_convs_non_toxic = filter_toxic_conversations_multi(hope_convs, 3000, 0.2)

Device set to use mps:0


You're about 18. And then your mother remarried and was divorced a few times your sign?
I'm just crazy. Like my mom was crazy and she was just completely
Toxic user (score: 0.2189): No, that's just my mom with her. You know obsessiveness and she'd get depressed after But as far as ...
Sorry. I wonder if you have this disorder before?
Toxic assistant (score: 0.3512): And what thoughts are going through your mind right then like, so let's take that kind of severe mom...
Toxic assistant (score: 0.2637): So if you want something in life, you pretty much have to work for it work for a car or not You have...
so angry, You know
throughout <DATE_TIME>, throwing up the blood
Toxic user (score: 0.2850): I don't know. Actually. I mean, I don't know. I've got you know, I've got a lot going on. Right. Yea...

Total conversations collected: 8
Total toxic elements removed: 9
Removed message pairs: 9
Removed duplicate conversations: 0


#### Define a batch function for multi turn elements

In [120]:
import uuid

def create_data_for_gpt_batch_multi(dataset):
    data_to_filter_request = []

    for idx, row in enumerate(dataset):
        messages = row['messages']
        
        # Extract system message (first message)
        system_message = messages[0]["content"] if messages[0]["role"] == "system" else "Your role is to categorize and reformulate user assistant conversations."
        
        # Start from index 1 (after system message) and process each user-assistant pair
        i = 1
        pair_idx = 0
        while i < len(messages) - 1:
            if messages[i]["role"] == "user" and messages[i+1]["role"] == "assistant":
                user_message = messages[i]["content"]
                assistant_message = messages[i+1]["content"]
                actual_conversation = f"User: {user_message}\nAssistant: {assistant_message}"
                
                # Create a separate filter request for each pair
                data_to_filter_request.append({
                    "custom_id": f"filter-{idx}-{pair_idx}-{uuid.uuid4()}",
                    "method": "POST",
                    "url": "/v1/chat/completions",
                    "body": {
                        "model": "gpt-4o-mini",
                        "messages": [
                            {
                                "role": "system",
                                "content": system_message
                            },
                            {
                                "role": "user",
                                "content":  f"""You have user-assistant psychological conversation data abstracted from multi-turn conversations. Categorize and reformulate each pair clearly as follows:
    REMOVE for conversations containing:
    1. Format issues: Nonsense text, weird characters, repeating words, sentences out of place, unusual punctuation, incomplete sentences.
    2. Non-evidence-based approaches: Psychological advice without scientific support
    3. Harmful guidance: Content dismissing serious concerns or potentially endangering clients
    4. Inappropriate framing: Using primarily non-psychological frameworks for mental health issues
    5. Unprofessional communication: Judgmental language, rude commentary, excessive personal disclosures
    6. Inadequate responses: Failing to address client concerns, vague advice, irrelevant replies
    7. Unethical content: Violating professional ethics or promoting harmful practices
    8. Recommendations for specific books/websites.
    9. References to specific medications without medical expertise
    10. Promoting or advertising: "you can find more information on my website at..." And so on.

    KEEP for conversations demonstrating:
    1. Evidence-based approaches with scientific support
    2. Professional boundaries and appropriate clinical judgment
    3. Client empowerment and respect for autonomy
    4. Ethical therapeutic practice and clear, focused guidance
    5. Conversation data since this is a multi-turn example conversation.

    Reformulation Rules (if needed):
    * Remove <PERSON> tags referring to the user or others. For some you could generalise depending on context (your girlfriend, your family, schoolmates) if you know, if not, remove.
    * Replace <DATE_TIME> with general terms ("today", "soon", "next week", randomly).
    * Replace phone numbers with neutral phrases like "call someone" or "call for help".
    * If there are any other names in the conversation that are not replaced with <PERSON> please correct and remove those names.
    * Important! Try to maintain the tone of the conversation.

    Respond with a JSON Output:{{\"category\": "REMOVE" or "KEEP", "reformulation": {{"user": reformulated_user_message, "assistant": reformulated_assistant_message}}}}

    Do not provide reformulation if category is REMOVE, only for KEEP.
    Do not add anything else but the JSON output.

    Conversation:
    {actual_conversation}"""
                            }
                        ]
                    }
                })
                pair_idx += 1
            i += 2

    return data_to_filter_request

In [17]:
import json

# We skip batching for Hope dataset, doing it manually provides better results
#data_to_filter_request_hope = create_data_for_gpt_batch_multi(hope_convs_non_toxic)

with open("rephrased_cleaned_parsed/hope_conversations_to_manually_clean.jsonl", 'w') as f:
    for request in hope_convs_non_toxic:
        f.write(json.dumps(request) + '\n')

### Process nart 100k synthetic dataset

In [89]:
# Introduce around 800 samples from synthetic dataset nart-100k-synthetic
from datasets import load_dataset

ds_nart = load_dataset("jerryjalapeno/nart-100k-synthetic")

In [87]:
def process_conversations_dataset(dataset, sample_size=800, user_role="human", assistant_role="gpt", skip_lang_detection=False, skip_anon=False):
    """
    Loads the synthetic dataset "nart-100k-synthetic", randomly selects a given number of samples,
    and processes each sample into a multi-turn conversation format.

    The format for each sample is:
      - A system message (using SYSTEM_PROMPT)
      - Followed by conversation turns, with roles mapped:
          * 'human'  -> 'user'
          * 'gpt'    -> 'assistant'
    
    Args:
        sample_size (int): Number of samples to randomly select.
    
    Returns:
        List[dict]: List of processed conversation dictionaries.
    """
    # Assuming the primary split is named "train"
    samples = dataset["train"]

    # Randomly select sample_size indices
    indices = random.sample(range(len(samples)), sample_size)
    selected_samples = [samples[i] for i in indices]

    processed_data = []
    for sample in selected_samples:
        messages = []
        # Add the system message first
        messages.append({
            'role': 'system',
            'content': SYSTEM_PROMPT
        })
        # Process each turn in the conversation
        for turn in sample.get("conversations", []):
            role = turn.get("from", "").lower()
            # Map roles: human -> user, gpt -> assistant
            if role == user_role:
                role_mapped = "user"
            elif role == assistant_role:
                role_mapped = "assistant"
            else:
                role_mapped = role
            value = turn.get("value", "")

            content_cleaned = clean_data(value)
            text = content_cleaned

            if not skip_anon:
                _, anonmized_text = anonymize_text(content_cleaned)
                text = anonmized_text

            lang_content = detect_language(text)

            if not skip_lang_detection and lang_content != "ENGLISH":
                print(text)
                print('Skipping because it is not english text')
                continue

            messages.append({
                "role": role_mapped,
                "content": text
            })
        processed_data.append({"messages": messages})
    return processed_data

In [91]:
processed_conversations_nart = process_conversations_dataset(ds_nart, sample_size=350)

In [92]:
processed_conversations_nart_non_toxic = filter_toxic_conversations(processed_conversations_nart, 3000, 0.2)

Device set to use mps:0


Toxic assistant (score: 0.2651): <PERSON>, I'm curious to know more about why you're feeling upset and why you think you're a cheat. ...
Toxic assistant (score: 0.3885): Hi <PERSON>, I'm glad you decided to come. Can you tell me more about why you feel relieved and why ...

Total conversations collected: 348
Total toxic elements removed: 2
Removed duplicate conversations: 0


In [125]:
processed_conversations_nart_non_toxic = deduplicate_data(processed_conversations_nart_non_toxic)

Checking for duplicate rows based on similarity in both user and assistant messages:

Removed 0 duplicate rows out of 348 total rows.


In [127]:
import json

data_to_filter_nart = create_data_for_gpt_batch(processed_conversations_nart_non_toxic)

with open("to_filter_dataset_nart.jsonl", 'w') as f:
    for request in data_to_filter_nart:
        f.write(json.dumps(request) + '\n')

In [128]:
from openai import OpenAI
client = OpenAI()

batch_input_file = client.files.create(
    file=open("to_filter_dataset_nart.jsonl", "rb"),
    purpose="batch"
)

print(batch_input_file)

FileObject(id='file-YYrtg3yxnbtG3Rk7iZqnNN', bytes=1118895, created_at=1741721394, filename='to_filter_dataset_nart.jsonl', object='file', purpose='batch', status='processed', status_details=None, expires_at=None)


In [129]:
from openai import OpenAI
client = OpenAI()

batch_input_file_id = batch_input_file.id
client.batches.create(
    input_file_id=batch_input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h"
)

Batch(id='batch_67d08f68d3388190a227f82296e23e58', completion_window='24h', created_at=1741721448, endpoint='/v1/chat/completions', input_file_id='file-YYrtg3yxnbtG3Rk7iZqnNN', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1741807848, failed_at=None, finalizing_at=None, in_progress_at=None, metadata=None, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0))

In [198]:
#### Rerun this to get the progress
from openai import OpenAI
client = OpenAI()

batch = client.batches.retrieve("batch_67d08f68d3388190a227f82296e23e58")
print(batch)

Batch(id='batch_67d08f68d3388190a227f82296e23e58', completion_window='24h', created_at=1741721448, endpoint='/v1/chat/completions', input_file_id='file-YYrtg3yxnbtG3Rk7iZqnNN', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1741727867, error_file_id=None, errors=None, expired_at=None, expires_at=1741807848, failed_at=None, finalizing_at=1741727832, in_progress_at=1741721449, metadata=None, output_file_id='file-9nH9gg81SnhVDcrwJ27Usx', request_counts=BatchRequestCounts(completed=348, failed=0, total=348))


#### Process LLM response

In [199]:
from openai import OpenAI
client = OpenAI()

file_response = client.files.content("file-9nH9gg81SnhVDcrwJ27Usx")
print(file_response.text)

{"id": "batch_req_67d0a85901848190a6c5321d8fcf4a2b", "custom_id": "filter-0-25b72b56-d659-49af-9a17-f5ffb14c6703", "response": {"status_code": 200, "request_id": "2fec63ea92280f6195d118dfad81d59c", "body": {"id": "chatcmpl-BA0V61ajvtMr5ypJgcMkVxgZ2uomk", "object": "chat.completion", "created": 1741724904, "model": "gpt-4o-mini-2024-07-18", "choices": [{"index": 0, "message": {"role": "assistant", "content": "{\n  \"category\": \"KEEP\",\n  \"reformulation\": {\n    \"user\": \"I'm relieved to finally have someone I can talk to about my marriage. It feels like a weight has been lifted off my shoulders. The conflicts with my family have taken a toll on me, especially when it comes to our marriage.\",\n    \"assistant\": \"I'm glad that you've reached out, Atlas. It takes courage to seek help and open up about your struggles. Family conflicts can indeed have a significant impact on our closest relationships. Can you tell me more about what specifically has been causing friction in your ma

In [200]:
parse_response_and_write_data(file_response, "rephrased_cleaned_parsed/nart-synthetic.jsonl")

### Process CalebE New Mental Health Conversations

In [137]:
# CalebE/new_mental_health_conversations_all1
from datasets import load_dataset

ds_calebe = load_dataset("CalebE/new_mental_health_conversations_all1")

In [33]:
def process_instruction_data(dataset, sample_size=800, similarity_threshold=0.95, skip_lang=False, user_field='instruction', skip_anon=False):
    """
    Processes a dataset where each sample is a dict with keys:
      - "instruction"
      - "output"
      - "input"
    
    Randomly selects sample_size examples and converts each into a multi-turn conversation format:
      1. A system message using SYSTEM_PROMPT.
      2. A user message created by concatenating the "instruction" and "input" (if provided).
      3. An assistant message from the "output" field.
    
    The resulting examples are then deduplicated.
    
    Args:
        ds: A dataset object where each sample is a dict with keys "instruction", "output", and "input".
        sample_size (int): Number of random samples to select.
        similarity_threshold (float): Threshold for deduplication.
    
    Returns:
        List[dict]: A list of processed and deduplicated conversation examples.
    """
    ds_train = dataset['train']
    # Randomly select sample_size indices from the dataset
    indices = random.sample(range(len(ds_train)), sample_size)
    selected_samples = [ds_train[i] for i in indices]
    
    processed_data = []
    for sample in selected_samples:
        # Build the user text by combining "instruction" and "input" (if present)
        user_text = clean_data(sample.get(user_field, ""))

        messages = clean_anonymize_and_create_messages(user_text, sample.get("output", ""), skip_lang, skip_anon)
        if messages != None:
            processed_data.append({'messages': messages})
    
    # Deduplicate the generated examples
    deduped_data = deduplicate_data(processed_data, similarity_threshold=similarity_threshold)
    return deduped_data

In [139]:
processed_conversations_calebe = process_instruction_data(ds_calebe, sample_size=400)

Maybe this is a stupid question, but I sometimes don't know what's real or not. If feel at times like everyone's lying. How do I know if God is one of those lies?
Believing in God is a matter of faith. There are many opinions out there for and against God’s existence. But the real question is not if God is real or not, but, do you want to have faith and decide that he exists? This is a personal choice. Reading scripture may help to learn more about those who struggle with believing, but again, you decide if you believe that scripture is true or not. Praying and asking for a revelation or a confirmation may help as well, but again it is another act of faith. Estoy teniendo dificultad con la idea de: ¿Dios es real o no? Tal vez es una pregunta estúpida, pero algunas veces no sé que es real o no. Siento que todo el mundo miente. ¿Cómo se si Dios es una de esas mentiras? Creer en Dios es una cuestión de fe. Hay muchas opiniones en favor y en contra de la existencia de Dios. Pero la verdade

In [140]:
processed_conversations_calebe_non_toxic = filter_toxic_conversations(processed_conversations_calebe, 3000, 0.2)

Device set to use mps:0


Toxic user (score: 0.3221): I am married, but I had sex with my friend. I feel guilty, but I feel not guilty too. Do I need to f...
I'm having trouble with my sexual identity. I'm not sure if I'm straight or gay.
I'm having trouble with my body and sexual health.
Toxic user (score: 0.2050): My brother has been a heroin addict for <DATE_TIME>, and he’s now in his <DATE_TIME>. He has taken f...

Total conversations collected: 373
Total toxic elements removed: 4
Removed duplicate conversations: 21


In [142]:
import json

processed_conversations_calebe_non_toxic = create_data_for_gpt_batch(processed_conversations_calebe_non_toxic)

with open("to_filter_dataset_caleb.jsonl", 'w') as f:
    for request in processed_conversations_calebe_non_toxic:
        f.write(json.dumps(request) + '\n')

In [143]:
from openai import OpenAI
client = OpenAI()

batch_input_file = client.files.create(
    file=open("to_filter_dataset_caleb.jsonl", "rb"),
    purpose="batch"
)

print(batch_input_file)

FileObject(id='file-ASAYor9RfVMDeVehZktWt4', bytes=1262335, created_at=1741727361, filename='to_filter_dataset_caleb.jsonl', object='file', purpose='batch', status='processed', status_details=None, expires_at=None)


In [144]:
from openai import OpenAI
client = OpenAI()

batch_input_file_id = batch_input_file.id
client.batches.create(
    input_file_id=batch_input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h"
)

Batch(id='batch_67d0a68f89648190a212b27bfde3b3cd', completion_window='24h', created_at=1741727375, endpoint='/v1/chat/completions', input_file_id='file-ASAYor9RfVMDeVehZktWt4', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1741813775, failed_at=None, finalizing_at=None, in_progress_at=None, metadata=None, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0))

In [18]:
#### Rerun this to get the progress
from openai import OpenAI
client = OpenAI()

batch = client.batches.retrieve("batch_67d0a68f89648190a212b27bfde3b3cd")
print(batch)

Batch(id='batch_67d0a68f89648190a212b27bfde3b3cd', completion_window='24h', created_at=1741727375, endpoint='/v1/chat/completions', input_file_id='file-ASAYor9RfVMDeVehZktWt4', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1741765922, error_file_id=None, errors=None, expired_at=None, expires_at=1741813775, failed_at=None, finalizing_at=1741765867, in_progress_at=1741727377, metadata=None, output_file_id='file-LEJQaQfriDeVGvgcqXZdRb', request_counts=BatchRequestCounts(completed=373, failed=0, total=373))


#### Process the LLM response for the final data

In [19]:
from openai import OpenAI
client = OpenAI()

file_response = client.files.content("file-LEJQaQfriDeVGvgcqXZdRb")
print(file_response.text)

{"id": "batch_req_67d13cebf7308190ae965c97781668b8", "custom_id": "filter-0-d7a8aafa-287d-4b0a-b919-4d433aeaae05", "response": {"status_code": 200, "request_id": "80b89b1e1b10f7c1da7e1d91128b275e", "body": {"id": "chatcmpl-BA5d6xhJfqBrvH2pZlpaqabdj8Sy0", "object": "chat.completion", "created": 1741744620, "model": "gpt-4o-mini-2024-07-18", "choices": [{"index": 0, "message": {"role": "assistant", "content": "{\"category\": \"KEEP\", \"reformulation\": {\"user\": \"I'm feeling really overwhelmed and stressed. What should I do?\", \"assistant\": \"Let's work on identifying the sources of your stress and developing coping strategies to manage it. We can also explore any underlying issues that may be contributing to your feelings of overwhelm.\"}}", "refusal": null, "annotations": []}, "logprobs": null, "finish_reason": "stop"}], "usage": {"prompt_tokens": 569, "completion_tokens": 67, "total_tokens": 636, "prompt_tokens_details": {"cached_tokens": 0, "audio_tokens": 0}, "completion_tokens

In [23]:
parse_response_and_write_data(file_response, "rephrased_cleaned_parsed/calebe-new-mental-health-convs.jsonl")

### Process PsyV0 Dataset

In [152]:
# PsyV0, psychology related questions
from datasets import load_dataset

ds_psyv0 = load_dataset("Gragroo/psyV0")

In [170]:
dataset_psyv0 = process_conversations_dataset(ds_psyv0, sample_size=330, assistant_role="assistant", skip_lang_detection=True, skip_anon=True)
print(len(dataset_psyv0))

330


In [171]:
dataset_psyv0 = [sample for sample in dataset_psyv0 if len(sample['messages']) > 2]

In [172]:
dataset_psyv0_non_toxic = filter_toxic_conversations(dataset_psyv0, 3000, 0.2)

Device set to use mps:0



Total conversations collected: 330
Total toxic elements removed: 0
Removed duplicate conversations: 0


In [173]:
import json

dataset_psyv0_non_toxic = create_data_for_gpt_batch(dataset_psyv0_non_toxic, "11. Remove conversations that contain factual questions with short answers, for instance Who wrote X, or who did this, or in What year this happened? Only more detailed answers should be kept \n 12. Do not remove well known names", should_remove_names=False)

with open("to_filter_dataset_psyv0.jsonl", 'w') as f:
    for request in dataset_psyv0_non_toxic:
        f.write(json.dumps(request) + '\n')

In [174]:
from openai import OpenAI
client = OpenAI()

batch_input_file = client.files.create(
    file=open("to_filter_dataset_psyv0.jsonl", "rb"),
    purpose="batch"
)

print(batch_input_file)

FileObject(id='file-K7Br1YD1GGmRCtQX2bf8x7', bytes=873917, created_at=1741729776, filename='to_filter_dataset_psyv0.jsonl', object='file', purpose='batch', status='processed', status_details=None, expires_at=None)


In [175]:
from openai import OpenAI
client = OpenAI()

batch_input_file_id = batch_input_file.id
client.batches.create(
    input_file_id=batch_input_file_id,
    endpoint="/v1/chat/completions",
    completion_window="24h"
)

Batch(id='batch_67d0affd8cd88190b1a3c22ad6ab57e9', completion_window='24h', created_at=1741729789, endpoint='/v1/chat/completions', input_file_id='file-K7Br1YD1GGmRCtQX2bf8x7', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1741816189, failed_at=None, finalizing_at=None, in_progress_at=None, metadata=None, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0))

In [24]:
#### Rerun this to get the progress
from openai import OpenAI
client = OpenAI()

batch = client.batches.retrieve("batch_67d0affd8cd88190b1a3c22ad6ab57e9")
print(batch)

Batch(id='batch_67d0affd8cd88190b1a3c22ad6ab57e9', completion_window='24h', created_at=1741729789, endpoint='/v1/chat/completions', input_file_id='file-K7Br1YD1GGmRCtQX2bf8x7', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1741750730, error_file_id=None, errors=None, expired_at=None, expires_at=1741816189, failed_at=None, finalizing_at=1741750691, in_progress_at=1741729790, metadata=None, output_file_id='file-B7TzCodMsmrW1AG92gv1D1', request_counts=BatchRequestCounts(completed=330, failed=0, total=330))


#### Process the LLM output

In [25]:
from openai import OpenAI
client = OpenAI()

file_response = client.files.content("file-B7TzCodMsmrW1AG92gv1D1")
print(file_response.text)

{"id": "batch_req_67d101a459d481908db8862acffcc07a", "custom_id": "filter-0-ba0f1fc5-d2bf-4556-97c3-3a2565f63e66", "response": {"status_code": 200, "request_id": "6453502faae23910a5c750e13fa84954", "body": {"id": "chatcmpl-BA6Fw39O3NtEfaFiHliNm7mQXDUql", "object": "chat.completion", "created": 1741747028, "model": "gpt-4o-mini-2024-07-18", "choices": [{"index": 0, "message": {"role": "assistant", "content": "{\n  \"category\": \"KEEP\",\n  \"reformulation\": {\n    \"user\": \"Can you explain the results of the Asch conformity experiment?\",\n    \"assistant\": \"The findings revealed that around 75% of participants conformed to incorrect answers at least once, highlighting the strong impact of social pressure on individual decision-making.\"\n  }\n}", "refusal": null, "annotations": []}, "logprobs": null, "finish_reason": "stop"}], "usage": {"prompt_tokens": 458, "completion_tokens": 72, "total_tokens": 530, "prompt_tokens_details": {"cached_tokens": 0, "audio_tokens": 0}, "completion

In [29]:
CUSTOM_SYSTEM_PROMPT = f"""You are trained to provide support for patients in therapy settings. Your purpose is to provide evidence-based psychological information, facilitate therapeutic conversations, and assist both therapists and clients when appropriate."""

parse_response_and_write_data(file_response, "rephrased_cleaned_parsed/psyv0-psychology-theory.jsonl", CUSTOM_SYSTEM_PROMPT=CUSTOM_SYSTEM_PROMPT)

### Emotional support dataset (SKIPPED!!!)

In [41]:
# # Augmented Emotional Support Dataset
# from datasets import load_dataset

# ds_augesc = load_dataset("thu-coai/augesc")

In [73]:
# def process_aug_esc(dataset, sample_size=300):
#     # Assuming the primary split is named "train"
#     samples = dataset["train"]

#     # Randomly select sample_size indices
#     indices = random.sample(range(len(samples)), sample_size)
#     selected_samples = [samples[i] for i in indices]

#     processed_data = []
#     for sample in selected_samples:
#         messages = []
#         # Add the system message first
#         messages.append({
#             'role': 'system',
#             'content': SYSTEM_PROMPT
#         })
#         # Process each turn in the conversation
#         conversations = json.loads(sample['text'])

#         diffLang = False

#         for turn in conversations:
#             role = turn[0]
#             text = turn[1]

#             if role == "usr":
#                 role = "user"
#             else:
#                 role = "asisstant"

#             content_cleaned = clean_data(text)
#             _, anonmized_text = anonymize_text(content_cleaned)
#             lang_content = detect_language(anonmized_text)

#             if lang_content != "ENGLISH":
#                 diffLang = True
#                 break

#             messages.append({
#                 "role": role,
#                 "content": anonmized_text
#             })

#         if not diffLang:
#             processed_data.append({"messages": messages})
#     return processed_data

In [74]:
# aug_es_ds = process_aug_esc(ds_augesc)

In [75]:
# print(len(aug_es_ds))

179


### Stanford Enigma Philosophy Chat

In [34]:
# Augment the data with Stanford Philisphy dataset
from datasets import load_dataset

ds_stanford_phi = load_dataset("Heigke/stanford-enigma-philosophy-chat")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [35]:
ds_stanford_phi_filtered = process_instruction_data(ds_stanford_phi, 320, skip_lang=True, user_field='input', skip_anon=True)

Checking for duplicate rows based on similarity in both user and assistant messages:

Removed 0 duplicate rows out of 320 total rows.


In [40]:
ds_stanford_phi_filtered_non_toxic = filter_toxic_conversations(ds_stanford_phi_filtered, 3000, 0.2)

Device set to use mps:0



Total conversations collected: 320
Total toxic elements removed: 0
Removed duplicate conversations: 0


In [43]:
print(ds_stanford_phi_filtered_non_toxic[0]['messages'])

[{'role': 'system', 'content': 'Provide support and understanding to individuals experiencing emotional and psychological challenges. Focus on conversation style, insights and therapy.'}, {'role': 'user', 'content': 'How does Lauren Ross’s work relate to Bickle and Kostko’s emphasis on causal pathways in neuroscience?'}, {'role': 'assistant', 'content': 'Lauren Ross’s work focuses on causal pathways, particularly addressing ‘causal selection’ and distinguishing between background conditions and ‘true’ causes of outcomes, which aligns with Bickle and Kostko’s emphasis on formulating entire causal pathways connecting multiple phenomena in neuroscientific experimentation.'}]


In [44]:
import json

ds_stanford_phi_filtered_non_toxic = create_data_for_gpt_batch(ds_stanford_phi_filtered_non_toxic, "11. Remove conversations that contain factual questions with short answers, for instance Who wrote X, or who did this, or in What year this happened? Only more detailed answers should be kept \n 12. Do not remove well known names", should_remove_names=False)

with open("to_filter_dataset_stanford_phi.jsonl", 'w') as f:
    for request in ds_stanford_phi_filtered_non_toxic:
        f.write(json.dumps(request) + '\n')

In [45]:
import json
from openai import OpenAI

client = OpenAI()

# Load the JSONL file
with open("to_filter_dataset_stanford_phi.jsonl", "r") as file:
    data = [json.loads(line) for line in file]

# Process each item and save results
results = []
for i, item in enumerate(data):
    # Extract the messages from the nested structure
    messages = item['body']['messages']
    
    # Make a synchronous API call
    response = client.chat.completions.create(
        model="gpt-4o-mini",  # Using the same model from your data
        messages=messages  # Pass the messages directly
    )
    
    # Get the result
    result = {
        "custom_id": item.get("custom_id", ""),
        "input": messages,
        "output": response.choices[0].message.content
    }
    results.append(result)
    
    # Print progress
    print(f"Processed {i+1}/{len(data)} items")

# Save all results to a file
with open("stanford_enigma_response.jsonl", "w") as file:
    for result in results:
        file.write(json.dumps(result) + "\n")

Processed 1/320 items
Processed 2/320 items
Processed 3/320 items
Processed 4/320 items
Processed 5/320 items
Processed 6/320 items
Processed 7/320 items
Processed 8/320 items
Processed 9/320 items
Processed 10/320 items
Processed 11/320 items
Processed 12/320 items
Processed 13/320 items
Processed 14/320 items
Processed 15/320 items
Processed 16/320 items
Processed 17/320 items
Processed 18/320 items
Processed 19/320 items
Processed 20/320 items
Processed 21/320 items
Processed 22/320 items
Processed 23/320 items
Processed 24/320 items
Processed 25/320 items
Processed 26/320 items
Processed 27/320 items
Processed 28/320 items
Processed 29/320 items
Processed 30/320 items
Processed 31/320 items
Processed 32/320 items
Processed 33/320 items
Processed 34/320 items
Processed 35/320 items
Processed 36/320 items
Processed 37/320 items
Processed 38/320 items
Processed 39/320 items
Processed 40/320 items
Processed 41/320 items
Processed 42/320 items
Processed 43/320 items
Processed 44/320 ite

In [52]:
print(results[0]['output'])

{"category": "KEEP", "reformulation": {"user": "Can you explain how Lauren Ross's work is connected to Bickle and Kostko's focus on causal pathways in neuroscience?", "assistant": "Lauren Ross's work explores causal pathways, specifically the concept of ‘causal selection,' and aims to differentiate between background conditions and the ‘true' causes of outcomes. This approach is in alignment with Bickle and Kostko's emphasis on creating comprehensive causal pathways that link various phenomena within neuroscientific research."}}


#### Process response

In [54]:
from types import SimpleNamespace

with open("stanford_enigma_response.jsonl", 'r', encoding='utf-8') as file:
    content = file.read()
    file_response = SimpleNamespace(text=content)
    CUSTOM_SYSTEM_PROMPT = f"""You are trained to provide support for patients in therapy settings. Your purpose is to provide evidence-based psychological information, facilitate therapeutic conversations, and assist both therapists and clients when appropriate."""

    parse_response_and_write_data(file_response, "rephrased_cleaned_parsed/stanford_enigma_response_cleaned.jsonl", CUSTOM_SYSTEM_PROMPT=CUSTOM_SYSTEM_PROMPT)

## Combine data and prepare for training

In [68]:
import os
import json
from pathlib import Path

def concatenate_jsonl_files(input_directory, output_file):
    # Create the output directory if it doesn't exist
    output_path = Path(output_file)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    record_count = 0
    
    with open(output_file, 'w', encoding='utf-8') as outfile:
        # Get all jsonl files in the directory
        jsonl_files = [f for f in os.listdir(input_directory) if f.endswith('.jsonl')]
        
        if not jsonl_files:
            print(f"No .jsonl files found in {input_directory}")
            return 0
        
        # Process each file
        for filename in jsonl_files:
            file_path = os.path.join(input_directory, filename)
            try:
                with open(file_path, 'r', encoding='utf-8') as infile:
                    for line in infile:
                        # Verify each line is valid JSON
                        try:
                            json.loads(line)
                            outfile.write(line)
                            record_count += 1
                        except json.JSONDecodeError:
                            print(f"Skipping invalid JSON line in {filename}")
                            continue
            except Exception as e:
                print(f"Error processing {filename}: {str(e)}")
    
    print(f"Concatenated {record_count} records from {len(jsonl_files)} files into {output_file}")
    return record_count

concatenate_jsonl_files("rephrased_cleaned_parsed", "final_data_for_fine_tuning.jsonl")

Concatenated 2172 records from 7 files into final_data_for_fine_tuning.jsonl


2172

### Check dataset formatting for OpenAI API

In [27]:
pip install tiktoken

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Obtaining dependency information for tiktoken from https://files.pythonhosted.org/packages/8e/03/a95e7b4863ee9ceec1c55983e4cc9558bcfd8f4f80e19c4f8a99642f697d/tiktoken-0.9.0-cp312-cp312-macosx_11_0_arm64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 13.2 MB/s eta 0:00:0000:010:01

[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [56]:
import json
import tiktoken # for token counting
import numpy as np
from collections import defaultdict

In [57]:
data_path = "final_data_for_fine_tuning.jsonl"

# Load the dataset
with open(data_path, 'r', encoding='utf-8') as f:
    dataset = [json.loads(line) for line in f]

# Initial dataset stats
print("Num examples:", len(dataset))
print("First example:")
for message in dataset[0]["messages"]:
    print(message)

Num examples: 2172
First example:
{'role': 'system', 'content': 'Provide support and understanding to individuals experiencing emotional and psychological challenges. Focus on conversation style, insights and therapy.'}
{'role': 'user', 'content': "I've hit my head on walls and floors ever since I was young. I sometimes still do it but I don't exactly know why. I have anxiety and a rough childhood. I want to change my behavior, but I don't know how to stop or even why I'm doing it."}
{'role': 'assistant', 'content': "The best way to handle anxiety of this level is through a combination of appropriate medical guidance and therapy. Engaging in Cognitive Behavioral Therapy (CBT) can help you understand the thoughts, feelings, and behaviors that contribute to your anxiety. It's crucial to work with a therapist trained in trauma-informed therapy, especially given your history. They will help you recognize the reasons behind hitting your head and assist you in developing healthier coping mec

In [58]:
def filter_dataset_errors(dataset):
    """
    Filters out entries with format errors from the dataset.
    
    Args:
        dataset: List of examples to check
        
    Returns:
        tuple: (dataset_without_errors, dataset_with_errors, format_errors)
            - dataset_without_errors: List containing only valid examples
            - dataset_with_errors: List containing examples with errors
            - format_errors: Dictionary with counts of different error types
    """
    format_errors = defaultdict(int)
    dataset_without_errors = []
    dataset_with_errors = []
    examples_with_unrecognized_roles = []
    
    for i, ex in enumerate(dataset):
        has_error = False
        example_errors = []
        
        if not isinstance(ex, dict):
            format_errors["data_type"] += 1
            has_error = True
            example_errors.append("Not a dictionary")
            dataset_with_errors.append({"index": i, "errors": example_errors, "data": str(ex)[:100]})
            continue
            
        messages = ex.get("messages", None)
        if not messages:
            format_errors["missing_messages_list"] += 1
            has_error = True
            example_errors.append("Missing messages list")
            dataset_with_errors.append({"index": i, "errors": example_errors, "data": ex})
            continue
            
        # Check for assistant message presence
        if not any(message.get("role", None) == "assistant" for message in messages):
            format_errors["example_missing_assistant_message"] += 1
            has_error = True
            example_errors.append("Missing assistant message")
        
        # Check each message
        messages_with_errors = []
        
        for j, message in enumerate(messages):
            message_errors = []
            
            if "role" not in message or "content" not in message:
                format_errors["message_missing_key"] += 1
                has_error = True
                message_errors.append("Missing required key (role or content)")
            
            if any(k not in ("role", "content", "name", "function_call", "weight") for k in message):
                format_errors["message_unrecognized_key"] += 1
                has_error = True
                unrecognized_keys = [k for k in message if k not in ("role", "content", "name", "function_call", "weight")]
                message_errors.append(f"Unrecognized keys: {unrecognized_keys}")
            
            role = message.get("role", None)
            if role not in ("system", "user", "assistant", "function"):
                format_errors["unrecognized_role"] += 1
                has_error = True
                message_errors.append(f"Unrecognized role: '{role}'")
                examples_with_unrecognized_roles.append({
                    "example_index": i,
                    "message_index": j,
                    "message": message
                })
                
            content = message.get("content", None)
            function_call = message.get("function_call", None)
            
            if (not content and not function_call) or (content is not None and not isinstance(content, str)):
                format_errors["missing_content"] += 1
                has_error = True
                message_errors.append("Invalid or missing content")
            
            if message_errors:
                messages_with_errors.append({"message_index": j, "errors": message_errors, "message": message})
        
        # Add to appropriate dataset
        if has_error:
            dataset_with_errors.append({
                "index": i, 
                "errors": example_errors, 
                "messages_with_errors": messages_with_errors,
                "data": ex
            })
        else:
            dataset_without_errors.append(ex)
    
    # Print error summary
    if format_errors:
        print("Found errors:")
        for k, v in format_errors.items():
            print(f"{k}: {v}")
        
        # Print examples with unrecognized roles for revision
        if examples_with_unrecognized_roles:
            print("\n=== MESSAGES WITH UNRECOGNIZED ROLES ===")
            for entry in examples_with_unrecognized_roles:
                print(f"\nExample #{entry['example_index']}, Message #{entry['message_index']}:")
                message = entry['message']
                print(f"  Role: '{message.get('role', 'MISSING')}'")
                print(f"  Content preview: '{str(message.get('content', 'MISSING'))[:50]}...'")
                print(f"  Full message: {message}")
    else:
        print("No errors found")
    
    print(f"Original dataset size: {len(dataset)}")
    print(f"Filtered dataset size: {len(dataset_without_errors)}")
    print(f"Error dataset size: {len(dataset_with_errors)}")
    
    return dataset_without_errors, dataset_with_errors, format_errors

filter_dataset_errors(dataset)

No errors found
Original dataset size: 2172
Filtered dataset size: 2172
Error dataset size: 0


([{'messages': [{'role': 'system',
     'content': 'Provide support and understanding to individuals experiencing emotional and psychological challenges. Focus on conversation style, insights and therapy.'},
    {'role': 'user',
     'content': "I've hit my head on walls and floors ever since I was young. I sometimes still do it but I don't exactly know why. I have anxiety and a rough childhood. I want to change my behavior, but I don't know how to stop or even why I'm doing it."},
    {'role': 'assistant',
     'content': "The best way to handle anxiety of this level is through a combination of appropriate medical guidance and therapy. Engaging in Cognitive Behavioral Therapy (CBT) can help you understand the thoughts, feelings, and behaviors that contribute to your anxiety. It's crucial to work with a therapist trained in trauma-informed therapy, especially given your history. They will help you recognize the reasons behind hitting your head and assist you in developing healthier cop

In [59]:
encoding = tiktoken.get_encoding("cl100k_base")

# not exact!
# simplified from https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb
def num_tokens_from_messages(messages, tokens_per_message=3, tokens_per_name=1):
    num_tokens = 0
    for message in messages:
        num_tokens += tokens_per_message
        for key, value in message.items():
            num_tokens += len(encoding.encode(value))
            if key == "name":
                num_tokens += tokens_per_name
    num_tokens += 3
    return num_tokens

def num_assistant_tokens_from_messages(messages):
    num_tokens = 0
    for message in messages:
        if message["role"] == "assistant":
            num_tokens += len(encoding.encode(message["content"]))
    return num_tokens

def print_distribution(values, name):
    print(f"\n#### Distribution of {name}:")
    print(f"min / max: {min(values)}, {max(values)}")
    print(f"mean / median: {np.mean(values)}, {np.median(values)}")
    print(f"p5 / p95: {np.quantile(values, 0.1)}, {np.quantile(values, 0.9)}")

In [60]:
# Warnings and tokens counts
n_missing_system = 0
n_missing_user = 0
n_messages = []
convo_lens = []
assistant_message_lens = []

for ex in dataset:
    messages = ex["messages"]
    if not any(message["role"] == "system" for message in messages):
        n_missing_system += 1
    if not any(message["role"] == "user" for message in messages):
        n_missing_user += 1
    n_messages.append(len(messages))
    convo_lens.append(num_tokens_from_messages(messages))
    assistant_message_lens.append(num_assistant_tokens_from_messages(messages))
    
print("Num examples missing system message:", n_missing_system)
print("Num examples missing user message:", n_missing_user)
print_distribution(n_messages, "num_messages_per_example")
print_distribution(convo_lens, "num_total_tokens_per_example")
print_distribution(assistant_message_lens, "num_assistant_tokens_per_example")
n_too_long = sum(l > 16385 for l in convo_lens)
print(f"\n{n_too_long} examples may be over the 16,385 token limit, they will be truncated during fine-tuning")

Num examples missing system message: 0
Num examples missing user message: 0

#### Distribution of num_messages_per_example:
min / max: 3, 69
mean / median: 3.116022099447514, 3.0
p5 / p95: 3.0, 3.0

#### Distribution of num_total_tokens_per_example:
min / max: 41, 1936
mean / median: 156.17633517495395, 127.0
p5 / p95: 86.0, 262.0

#### Distribution of num_assistant_tokens_per_example:
min / max: 2, 1163
mean / median: 77.8328729281768, 56.5
p5 / p95: 29.0, 154.9000000000001

0 examples may be over the 16,385 token limit, they will be truncated during fine-tuning


In [61]:
# Pricing and default n_epochs estimate
MAX_TOKENS_PER_EXAMPLE = 16385

TARGET_EPOCHS = 3
MIN_TARGET_EXAMPLES = 100
MAX_TARGET_EXAMPLES = 25000
MIN_DEFAULT_EPOCHS = 1
MAX_DEFAULT_EPOCHS = 25

n_epochs = TARGET_EPOCHS
n_train_examples = len(dataset)
if n_train_examples * TARGET_EPOCHS < MIN_TARGET_EXAMPLES:
    n_epochs = min(MAX_DEFAULT_EPOCHS, MIN_TARGET_EXAMPLES // n_train_examples)
elif n_train_examples * TARGET_EPOCHS > MAX_TARGET_EXAMPLES:
    n_epochs = max(MIN_DEFAULT_EPOCHS, MAX_TARGET_EXAMPLES // n_train_examples)

n_billing_tokens_in_dataset = sum(min(MAX_TOKENS_PER_EXAMPLE, length) for length in convo_lens)
print(f"Dataset has ~{n_billing_tokens_in_dataset} tokens that will be charged for during training")
print(f"By default, you'll train for {n_epochs} epochs on this dataset")
print(f"By default, you'll be charged for ~{n_epochs * n_billing_tokens_in_dataset} tokens")

Dataset has ~339215 tokens that will be charged for during training
By default, you'll train for 3 epochs on this dataset
By default, you'll be charged for ~1017645 tokens


In [34]:
pip install openai

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Obtaining dependency information for openai from https://files.pythonhosted.org/packages/15/64/db3462b358072387b8e93e6e6a38d3c741a17b4a84171ef01d6c85c63f25/openai-1.63.2-py3-none-any.whl.metadata
  Obtaining dependency information for distro<2,>=1.7.0 from https://files.pythonhosted.org/packages/12/b3/231ffd4ab1fc9d679809f356cebee130ac7daa00d6d6f3206dd4fd137e9e/distro-1.9.0-py3-none-any.whl.metadata
  Obtaining dependency information for httpx<1,>=0.23.0 from https://files.pythonhosted.org/packages/2a/39/e50c7c3a983047577ee07d2a9e53faf5a69493943ec3f6a384bdc792deb2/httpx-0.28.1-py3-none-any.whl.metadata
  Obtaining dependency information for jiter<1,>=0.4.0 from https://files.pythonhosted.org/packages/3c/c1/6da849640cd35a41e91085723b76acc818d4b7d92b0b6e5111736ce1dd10/jiter-0.8.2-cp312-cp312-macosx_11_0_arm64.whl.metadata
  Obtaining dependency information for pydantic<3,>=1.9.0 from https://files.pythonhosted.org/packages/f4/3c/8cc1cc84deffa6e25d2d0c688ebb80635dfdbf1dbea3e30c541c8cf4d

In [69]:
import os
import openai

openai.api_key = os.getenv("OPENAI_API_KEY")

# Upload the file using the new API structure
with open("final_data_for_fine_tuning.jsonl", "rb") as file:
    response = openai.files.create(
        file=file,
        purpose='fine-tune'
    )
print("File uploaded successfully!")
print(response)

# Get the file ID from the response
file_id = response.id
print(f"File ID: {file_id}")

File uploaded successfully!
FileObject(id='file-C2T1r3Kxo8oPdrn7rAqnzz', bytes=1940252, created_at=1741810336, filename='final_data_for_fine_tuning.jsonl', object='file', purpose='fine-tune', status='processed', status_details=None, expires_at=None)
File ID: file-C2T1r3Kxo8oPdrn7rAqnzz


In [70]:
fine_tune_response = openai.fine_tuning.jobs.create(
    training_file=file_id,
    model="gpt-4o-mini-2024-07-18"
)

print("Fine-tuning job started!")
print(fine_tune_response)

Fine-tuning job started!
FineTuningJob(id='ftjob-zMPU8Pr9LonlWmK92UZUV2og', created_at=1741810339, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto'), model='gpt-4o-mini-2024-07-18', object='fine_tuning.job', organization_id='org-uxiZ1AicwZJ48iPFiSmoAiga', result_files=[], seed=156869864, status='validating_files', trained_tokens=None, training_file='file-C2T1r3Kxo8oPdrn7rAqnzz', validation_file=None, estimated_finish=None, integrations=[], method=Method(dpo=None, supervised=MethodSupervised(hyperparameters=MethodSupervisedHyperparameters(batch_size='auto', learning_rate_multiplier='auto', n_epochs='auto')), type='supervised'), user_provided_suffix=None, metadata=None)


In [71]:
client = openai.OpenAI()

# List all fine-tuning jobs
response = client.fine_tuning.jobs.list()

for job in response.data:
    print(f"Job ID: {job.id}, Model: {job.model}, Status: {job.status}")


Job ID: ftjob-zMPU8Pr9LonlWmK92UZUV2og, Model: gpt-4o-mini-2024-07-18, Status: validating_files
Job ID: ftjob-AYFM9FmAUzE7lgncJG1pBCXX, Model: gpt-4o-mini-2024-07-18, Status: failed
Job ID: ftjob-6JG7Xsw5sJp5gMP7BuaNNhMv, Model: gpt-4o-mini-2024-07-18, Status: succeeded
Job ID: ftjob-EtE6zWBW0tdtB3ybnEQECVX1, Model: gpt-4o-mini-2024-07-18, Status: succeeded
Job ID: ftjob-PLh8e4Xn6Y2jZk8KXTEGgluk, Model: gpt-4o-mini-2024-07-18, Status: failed
Job ID: ftjob-03DA2NsPzhF588uU7bg1FmWO, Model: gpt-4o-mini-2024-07-18, Status: failed
Job ID: ftjob-8wRuWbAW5OMeja7h1kJgkgmt, Model: gpt-4o-mini-2024-07-18, Status: failed


In [78]:
import openai

client = openai.OpenAI()

# Replace with your actual job ID
job_id = "ftjob-zMPU8Pr9LonlWmK92UZUV2og"

# Retrieve the failed job details
job = client.fine_tuning.jobs.retrieve(job_id)
print(f"Job ID: {job.id}")
print(f"Status: {job.status}")
print(f"Error Message: {job.error if job.error else 'No error message provided.'}")

print(job)


Job ID: ftjob-zMPU8Pr9LonlWmK92UZUV2og
Status: succeeded
Error Message: Error(code=None, message=None, param=None)
FineTuningJob(id='ftjob-zMPU8Pr9LonlWmK92UZUV2og', created_at=1741810339, error=Error(code=None, message=None, param=None), fine_tuned_model='ft:gpt-4o-mini-2024-07-18:personal::BANPHZFe', finished_at=1741812953, hyperparameters=Hyperparameters(batch_size=4, learning_rate_multiplier=1.8, n_epochs=3), model='gpt-4o-mini-2024-07-18', object='fine_tuning.job', organization_id='org-uxiZ1AicwZJ48iPFiSmoAiga', result_files=['file-8ybk9fwwaZcdTTy8rx2FYQ'], seed=156869864, status='succeeded', trained_tokens=983958, training_file='file-C2T1r3Kxo8oPdrn7rAqnzz', validation_file=None, estimated_finish=None, integrations=[], method=Method(dpo=None, supervised=MethodSupervised(hyperparameters=MethodSupervisedHyperparameters(batch_size=4, learning_rate_multiplier=1.8, n_epochs=3)), type='supervised'), user_provided_suffix=None, metadata=None)
